# **INSTALL DATASET FROM KAGGLE**

In [ ]:
!pip install kaggle -q

In [ ]:
# Configuring the path of kaggle.json file
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
# API to fetch the dataset from kaggle
!kaggle datasets download -d omkargurav/face-mask-dataset

# **EXTRACTING THE ZIPFILE**

In [ ]:
# Extract the compressed Dataset
from zipfile import ZipFile
import os

zip_path = "/content/face-mask-dataset.zip"
dataset_dir = "/content/data"

with ZipFile(zip_path, 'r') as zip:
  zip.extractall()

print("!!!Extracted!!!")
print(os.listdir(dataset_dir))

In [ ]:
!ls

# **IMPORTING DEPENDENCIES**

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import cv2
from google.colab.patches import cv2_imshow
from PIL import Image
from sklearn.model_selection import train_test_split

# **PREPROCESSING**

In [ ]:
with_mask_files = os.listdir('/content/data/with_mask' )
print(with_mask_files[0:5])
print(with_mask_files[-5:])

In [ ]:
without_mask_files = os.listdir('/content/data/without_mask' )
print(without_mask_files[0:5])
print(without_mask_files[-5:])

In [ ]:
print("Number of with mask images: ",len(with_mask_files))
print("Number of without mask images: ",len(without_mask_files))

# **Creating labels for the two classes of images**



*   with_mask    -> 1
*   without_mask -> 0




In [ ]:
with_mask_labels = [1]*3725

without_mask_labels = [0]*3828

In [ ]:
print(with_mask_labels[0:5])
print(without_mask_labels[0:5])

In [ ]:
print("Number of with mask images: ",len(with_mask_labels))
print("Number of without mask images: ",len(without_mask_labels))

In [ ]:
labels = with_mask_labels + without_mask_labels
print(len(labels))
print(with_mask_labels[0:5])
print(without_mask_labels[-5:0])

## **Displaying Images**

In [ ]:
img = mpimg.imread('/content/data/with_mask/with_mask_976.jpg')
imgplot = plt.imshow(img)
plt.show()

In [ ]:
img = mpimg.imread('/content/data/without_mask/without_mask_3504.jpg')
imgplot = plt.imshow(img)
plt.show()

## **Image Processing**

In [ ]:
with_mask_path = '/content/data/with_mask/'

data = []


for img_file in with_mask_files:

  image = Image.open(with_mask_path + img_file)
  image = image.resize((128,128))
  image = image.convert('RGB')
  image = np.array(image)
  data.append(image)


without_mask_path = '/content/data/without_mask/'


for img_file in without_mask_files:

  image = Image.open(without_mask_path + img_file)
  image = image.resize((128,128))
  image = image.convert('RGB')
  image = np.array(image)
  data.append(image)

In [ ]:
len(data)

In [ ]:
X = np.array(data)
y = np.array(labels)

print(X.shape)
print(y.shape)

# **Train Test Split**

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [ ]:
print(X.shape, X_train.shape, X_test.shape)

# **SCALING THE TRAINING DATA**

In [ ]:
# Scaling the data
X_train_scaled = X_train / 255.0

X_test_scaled = X_test / 255.0

# **BUILDING THE MODEL**

In [ ]:
import tensorflow as tf
from tensorflow import keras

In [ ]:
num_of_classes = 2

model = keras.Sequential()

model.add(keras.layers.Conv2D(64, kernel_size=(3,3), activation='relu', input_shape=(128,128,3)))
model.add(keras.layers.MaxPool2D(pool_size=(2,2)))

model.add(keras.layers.Conv2D(64, kernel_size=(3,3), activation='relu'))
model.add(keras.layers.MaxPool2D(pool_size=(2,2)))

model.add(keras.layers.Flatten())

model.add(keras.layers.Dense(128, activation = 'relu'))
model.add(keras.layers.Dropout(0.5))

model.add(keras.layers.Dense(128, activation = 'relu'))
model.add(keras.layers.Dropout(0.5))

model.add(keras.layers.Dense(num_of_classes, activation = 'sigmoid'))


# **COMPILING THE MODEL**

In [ ]:
model.compile(optimizer="adam",
              loss = "sparse_categorical_crossentropy",
              metrics = ["accuracy"])

# **TRAINING THE MODEL**

In [ ]:
history = model.fit(X_train_scaled, y_train, validation_split=0.1, epochs=5)

# **EVALUATING THE MODEL**

In [ ]:
loss, accuracy = model.evaluate(X_test_scaled, y_test)
print("Test Accuracy:", accuracy)

# **VISUALIZING THE LOSS AND ACCURACY METRICS**

In [ ]:
h = history
plt.plot(h.history["loss"], label="Training Loss")
plt.plot(h.history["val_loss"], label="Validation Loss")
plt.legend()
plt.show()

plt.plot(h.history["accuracy"], label="Training Accuracy")
plt.plot(h.history["val_accuracy"], label="Validation Accuracy")
plt.legend()
plt.show()

# **GETTING PREDICTIONS**

In [ ]:
input_img_path = input("Path of image to be predicted: ")
input_img = cv2.imread(input_img_path)

cv2_imshow(input_img)

input_img_resized = cv2.resize(input_img, (128,128))

input_img_scaled = input_img_resized / 255.0

input_img_reshaped = np.reshape(input_img_scaled, [1,128,128,3])

input_pred = model.predict(input_img_reshaped)
print(input_pred)

input_pred_label = np.argmax(input_pred)
print(input_pred_label)

if input_pred_label == 1:
  print("Person is wearing a mask")
else:
  print("No mask Detected!!")

In [ ]:
# No Mask #1021, #1008
# Mask